Ridership ≈ Service × Demand × Accessibility

In [1]:
pip install shared_utils

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Importing necessary package 
import pandas as pd 
import geopandas as gpd
pd.set_option('display.max_columns', None)
import os
import google.auth
import gcsfs
fs = gcsfs.GCSFileSystem()
import statsmodels.api as sm
import numpy as np
from scipy.stats import skew


import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [3]:
GCS_FILE_PATH = 'gs://calitp-analytics-data/data-analyses/ahsc_grant/ahsc_riderships/AHSC_2026'

In [4]:
with fs.open(f"{GCS_FILE_PATH}/stop_route_df.parquet", "rb") as f: 
    stop_route_df = gpd.read_parquet(f)

In [5]:
stop_route_df.head(2)

,organization_name,feed_key,stop_id,stop_name,stop_code,n_arrivals,n_routes,pt_geom,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date,geometry_x,geometry_y,total_pop_adj,poverty_pop_adj,non_us_citizen_adj,workers_with_no_car_adj,households_with_no_cars_adj,disabled_pop_adj,public_asst_pop_adj,inc_extremelylow_adj,inc_verylow_adj,inc_low_adj,male_seniors_adj,female_seniors_adj,veteran_pop_adj,male_youth_adj,inc_total_lowincome_adj,female_youth_adj,total_seniors_adj,jobs_tot_adj,total_youth_adj,ALAND_adj
0,Gold Coast Transit,3cb676436aad669e52042c0e97a9a051,VNACLR1,.,None,14.0,1.0,POINT(-119.294028 34.343645),Weekday,0.0,3.0,2025-05-01,2025-05-31,POINT (64936.582 -407800.219),"POLYGON ((65737.380 -407879.090, 65725.793 -40...",23.285073,2.446784,2.699588,0.352120,0.325034,3.114909,1.191791,5.372089,4.550476,1.977290,1.589055,1.877974,0.749384,0.993159,11.899855,2.248152,3.467029,4.207384,3.241311,1.092064e+06
1,Gold Coast Transit,3cb676436aad669e52042c0e97a9a051,VNACLR1,.,None,14.0,1.0,POINT(-119.294028 34.343645),Weekday,0.0,3.0,2025-05-01,2025-05-31,POINT (64936.582 -407800.219),"POLYGON ((65015.454 -408601.016, 64936.582 -40...",18.168282,1.502458,0.458582,0.072408,0.138781,2.594607,0.971469,3.843638,5.346097,1.574866,2.226535,2.974747,1.092148,0.886993,10.764601,0.524955,5.201282,2.594607,1.411949,8.775763e+05


In [14]:
# Convert n_arrivals and n_routes to integer
stop_route_df['n_arrivals'] = stop_route_df['n_arrivals'].fillna(0).astype(int)
stop_route_df['n_routes'] = stop_route_df['n_routes'].fillna(0).astype(int)

In [13]:
stop_route_df.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 107213 entries, 0 to 107212
Data columns (total 35 columns):
 #   Column                       Non-Null Count   Dtype   
---  ------                       --------------   -----   
 0   organization_name            107213 non-null  object  
 1   feed_key                     107213 non-null  object  
 2   stop_id                      107213 non-null  object  
 3   stop_name                    107213 non-null  object  
 4   stop_code                    103948 non-null  object  
 5   n_arrivals                   107213 non-null  float64 
 6   n_routes                     107213 non-null  float64 
 7   pt_geom                      107213 non-null  object  
 8   day_type                     107213 non-null  object  
 9   average_daily_boardings      107213 non-null  float64 
 10  average_daily_alightings     98658 non-null   float64 
 11  start_date                   107213 non-null  object  
 12  end_date                     107213 

In [6]:
# stop_route_df['is_route_level'] = stop_route_df['organization_name'].isin([
#     'Foothill Transit',
#     'Gold Coast Transit',
#     'Golden Gate Transit',
#     'Long Beach Transit',
#     'Orange County Transportation Authority',
#     'SacRT Bus',
#     'Samtrans',
#     'SDMTS',
#     'Culver City Bus',
#     'Big Blue Bus',
#     'Riverside Transit'
# ]).astype(int)

In [7]:
# stop_route_df['n_routes_effective'] = stop_route_df['n_routes'] * (1 - stop_route_df['is_route_level'])

### Log-linear Regression Model

In [8]:
# Log-linear regression
# Dependent variable: log('average_daily_boardings')
# Predictors: n_trips, n_routes, total_pop_adj, households_with_no_cars_adj, jobs_tot_adj

# Copy the dataframe
df = stop_route_df.copy()

# 2. Create log dependent variable
# Replace 0 with NaN BEFORE log (log(0) = -inf)
df['log_boardings'] = np.log(df['average_daily_boardings'].replace(0, np.nan))

# 3. Select only rows with all needed variables
df = df.dropna(subset=[
    'average_daily_boardings',
    'n_arrivals',
    'n_routes',
    'total_pop_adj',
    'households_with_no_cars_adj'
])

# 4. Define Y and X
y = df['average_daily_boardings']

X = df[['n_arrivals', 'n_routes',
        'total_pop_adj', 'households_with_no_cars_adj']]

# 5. Add constant
X = sm.add_constant(X)

# 6. Fit model
model = sm.OLS(y, X).fit()

# 7. Show results
print(model.summary())

                               OLS Regression Results                              
Dep. Variable:     average_daily_boardings   R-squared:                       0.064
Model:                                 OLS   Adj. R-squared:                  0.063
Method:                      Least Squares   F-statistic:                     1817.
Date:                     Fri, 03 Apr 2026   Prob (F-statistic):               0.00
Time:                             23:29:04   Log-Likelihood:            -8.1512e+05
No. Observations:                   107213   AIC:                         1.630e+06
Df Residuals:                       107208   BIC:                         1.630e+06
Df Model:                                4                                         
Covariance Type:                 nonrobust                                         
                                  coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------

- About 26.8% of the variation in dependent variable (here, stop-level ridership) is explained by the predictors in the model.
- Skew = 28.63 → Residuals are extremely asymmetric, violating normality.
- Kurtosis = 1793.542 → Residuals have ultra‑heavy tails, far from a normal distribution.
- JB statistic = 20 billion → The Jarque–Bera test overwhelmingly rejects normality.
- Durbin–Watson = 0.231 → Strong positive autocorrelation remains in the residuals.
- Condition number = 1.22e+04 → Predictors have scaling issues and possible multicollinearity.

### Checking Multicollinearity using VIF

VIF checks multicollinearity (correlation among predictors)

- Idea:
If a variable can be well explained by other predictors,
its coefficient becomes unstable and its standard error increases
- Interpretation:
VIF ≈ 1   → no correlation (good);
VIF > 5   → moderate concern;
VIF > 10  → serious multicollinearity
- Key point:
High VIF doesn't bias coefficients, but makes them noisy and hard to interpret

In [9]:
# Selecting only numeric regressors (Ridership ≈ Service × Demand × Accessibility)
X = stop_route_df[
    ['n_arrivals', 'n_routes',  'total_pop_adj', 'households_with_no_cars_adj' ]
].copy()

# Add constant
X = sm.add_constant(X)

vif_df = pd.DataFrame()
vif_df["variable"] = X.columns
vif_df["VIF"] = [
    variance_inflation_factor(X.values, i)
    for i in range(X.shape[1])
]

print(vif_df)

                      variable       VIF
0                        const  4.659402
1                   n_arrivals  1.468672
2                     n_routes  1.451546
3                total_pop_adj  1.382273
4  households_with_no_cars_adj  1.415487


All predictors have low VIFs (≈1–1.8), which means very little multicollinearity among chosen variables.
The model’s coefficients should therefore be stable and interpretable, with no inflation of standard errors.
The high VIF on the constant is not meaningful and can be ignored.

In [10]:
stop_route_df['average_daily_boardings'].describe()

count    107213.000000
mean         53.838787
std         501.035998
min           0.000000
25%           2.270677
50%           8.316155
75%          26.162374
max       41893.868537
Name: average_daily_boardings, dtype: float64

Data is highly skewed (28.63) .
Skewed data is data that isn’t evenly distributed around the center, values bunch up on one side and stretch into a long tail.
This means a few stops have very high boardings and push the distribution out into a long tail.
75% of stops board ≤ 20 people

Possible solution: Use a count model (Poisson or Negative Binomial)  
Other models to check:
- Poisson
- Negative Binomial (handles overdispersion better)
- Zero‑inflated Poisson (if many zeros)
- Zero‑inflated NB (most flexible)

### Poisson Model

In [17]:
# Copy the dataset
df = stop_route_df.copy()

# 2. Select needed variables (drop missing)
df = df.dropna(subset=[
    'average_daily_boardings',
    'n_arrivals', 'n_routes',
    'total_pop_adj', 'households_with_no_cars_adj'
])

# 3. Define Y and X
y = df['average_daily_boardings']

X = df[['n_arrivals', 'n_routes',
        'total_pop_adj', 'households_with_no_cars_adj', 'poverty_pop_adj']]

# 4. Add intercept
X = sm.add_constant(X)

# 5. Fit Poisson GLM
poisson_model = sm.GLM(y, X, family=sm.families.Poisson()).fit()

# 6. Output summary
print(poisson_model.summary())


                    Generalized Linear Model Regression Results                    
Dep. Variable:     average_daily_boardings   No. Observations:               107213
Model:                                 GLM   Df Residuals:                   107207
Model Family:                      Poisson   Df Model:                            5
Link Function:                         Log   Scale:                          1.0000
Method:                               IRLS   Log-Likelihood:            -8.7676e+06
Date:                     Fri, 03 Apr 2026   Deviance:                   1.7120e+07
Time:                             23:34:26   Pearson chi2:                 1.68e+08
No. Iterations:                         15   Pseudo R-squ. (CS):              1.000
Covariance Type:                 nonrobust                                         
                                  coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------

This Poisson model fits very poorly: the Pearson chi‑square is huge, meaning the data are heavily over‑dispersed and the Poisson assumptions break.
- The coefficient for n_trips (0.0163) means each extra trip increases expected ridership by about 1.6%.
- n_routes (–0.268) is negative, which is unrealistic and indicates model misspecification due to over‑dispersion.
- total_pop_adj is negative, also unrealistic — another sign the Poisson model is not appropriate.
- households_with_no_cars_adj is positive, meaning more zero‑car households predict higher ridership.
Overall: the Poisson model’s signs are distorted.

Pearson chi‑square = 4.77e+07 and Degrees of freedom ≈ 143325.
Hence, 
Dispersion estimate = Pearson χ² / df; 8.30e+07 / 151,044 ≈ 332.8

A valid Poisson model should have dispersion ≈ 1. Dispersion ≈ 550, which means:
Poisson model is severely overdispersed. >1 → overdispersion (variance is larger than the model assumes).

Variance >> mean:
- Poisson standard errors become too small
- p‑values become artificially tiny
- coefficients look falsely “precise”
- model fit is misleading

In [20]:
X = df[['n_arrivals', 'n_routes',
        'total_pop_adj', 'workers_with_no_car_adj',  'total_youth_adj',  'total_seniors_adj']]

# Same X and y as  Poisson
nb_model_extended = sm.GLM(
    y,
    X,
    family=sm.families.NegativeBinomial()
).fit()

print(nb_model_extended.summary())

/opt/conda/lib/python3.11/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


                    Generalized Linear Model Regression Results                    
Dep. Variable:     average_daily_boardings   No. Observations:               107213
Model:                                 GLM   Df Residuals:                   107207
Model Family:             NegativeBinomial   Df Model:                            5
Link Function:                         Log   Scale:                          1.0000
Method:                               IRLS   Log-Likelihood:            -4.5370e+05
Date:                     Fri, 03 Apr 2026   Deviance:                   2.2957e+05
Time:                             23:38:53   Pearson chi2:                 9.56e+05
No. Iterations:                         21   Pseudo R-squ. (CS):             0.7828
Covariance Type:                 nonrobust                                         
                              coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------

### Negative Binomial Model 

The Negative Binomial model behaves much better than the Poisson model because it corrects the strongly overdispersed nature of ridership data. After switching to a Negative Binomial model, the dispersion dropped to a reasonable level (≈5), the standard errors became realistic, the z‑values returned to interpretable ranges, and the overall fit stabilized. 


The NB model is used when:
- counts vary wildly
- some stops have way more riders than others
- data is highly skewed
- there are outliers or “hot spots”

Variance > Mean
This matches real transit ridership perfectly.

In the above model,
Pseudo R² (Cragg‑Uhler / Nagelkerke): 0.8978
Very high — the predictors explain a large proportion of the variation in ridership.
For transit ridership data (often noisy and over‑dispersed), this is exceptionally strong.



### Variables Interpretation

1. n_arrivals coefficient = 0.0483
Interpretation: Each additional trip serving the stop increases expected daily boardings by:
exp(0.0483) − 1 ≈ 4.95%

2. n_routes coefficient = 0.3658
Interpretation: For stops/agencies where n_routes_effective > 0 (aggregated stop-level data), each additional route increases expected daily boardings by:
exp(0.3658)−1 ≈ 44.1%



3. total_pop_adj – Population in stop buffer
total_pop_adj represents the estimated number of residents physically inside a stop’s 600 m buffer, after adjusting each census tract by the fraction of its area that overlaps the buffer.
Example: if 10% of a tract lies inside a stop’s buffer, 10% of the tract’s population is assigned to that stop.
Coefficient interpretation:
Model coefficient: 0.0001
Effect per additional person: exp(0.0001)−1≈0.01% increase in expected daily boardings per person

  This looks tiny for one person, but stop buffers usually contain hundreds or thousands of people.
     Scaling to realistic population changes:

    +1,000 residents in the stop buffer: exp(1000×0.0001)−1 ≈ 10.5% increase in ridership
    +10,000 residents in the stop buffer: exp(10,000×0.0001)−1 ≈ 171.8% increase in ridership

4. workers_with_no_car_adj (coef = 0.0006)
workers_with_no_car_adj is the estimated number of population that do not own a car within the stop catchment, computed the same area-weighted way.

For each additional zero-car household in the catchment:
exp(0.0006) – 1 ≈ 0.06%% increase in expected boardings

- +100 zero-car households in buffer : ≈ 6.18% ridership increase
- +1,000 zero-car households : ≈ 82.2% ridership increase

## Negative Binomial GLM (Log‑Link) Model Equation

The model estimates the expected daily boardings at stop *i* using a Negative Binomial
Generalized Linear Model with a log link.

The linear predictor is:

$$
\log(\mu_i) =
\beta_0
+ \beta_1 \, (\text{n\_trips}_i)
+ \beta_2 \, (\text{n\_routes}_i)
+ \beta_3 \, (\text{total\_pop\_adj}_i)
+ \beta_4 \, (\text{households\_with\_no\_cars\_adj}_i)
+ \beta_5 \, (\text{total\_youth\_adj}_i)
+ \beta_6 \, (\text{inc\_total\_lowincome\_adj}_i)
+ \beta_7 \, (\text{total\_seniors\_adj}_i)
$$

Where the expected value of daily boardings is:

$$
\mu_i = E[\text{average\_daily\_boardings}_i]
$$

The outcome follows a Negative Binomial distribution:

$$
Y_i \sim \text{NB}(\mu_i, \alpha)
$$

The log link implies that predictor effects are multiplicative:

$$
\mu_i = \exp\left(
\beta_0
+ \beta_1 \, \text{n\_trips}_i
+ \beta_2 \, \text{n\_routes}_i
+ \beta_3 \, \text{total\_pop\_adj}_i
+ \beta_4 \, \text{households\_with\_no\_cars\_adj}_i
+ \beta_5 \, \text{total\_youth\_adj}_i
+ \beta_6 \, \text{inc\_total\_lowincome\_adj}_i
+ \beta_7 \, \text{total\_seniors\_adj}_i
\right)
$$

Statsmodels’ `GLM(..., family=NegativeBinomial())` uses a fixed dispersion parameter $\alpha$
unless otherwise specified.

In log_linear ols regression model Y is continuous variable i.e. if n_trip has coefficient of 0.1 it means A one‑unit increase in trips increases the geometric mean of ridership by 10%.
While in this model: it means Each additional daily trip increases expected ridership by about 10%.

1) Log-linear OLS model
Modeling log(ridership). 
When the coefficient is 0.1, it means:
→ adding 1 trip increases ridership by about 10% on average (geometric mean).
2) Negative Binomial model
This model is built for count data (like number of riders).
A coefficient of 0.1 also means about a 10% increase, but interpreted as:
→ each extra trip increases the expected number of riders by 10%.

They sound almost identical, but:
- Log-linear OLS focuses on modeling a transformed version of ridership (log scale). Talks about the geometric mean (a bit abstract)
Negative Binomial
- directly models counts (actual riders). Talks about expected ridership (more natural and realistic for counts)